# Chapter 25: Graph SLAM

<a href="../lite/lab/index.html?path=ch25_graph_slam.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def draw_cov_ellipse(ax, mean, cov, n_std=2, **kwargs):
    from matplotlib.patches import Ellipse
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1,1], vecs[0,1]))
    w, h = 2 * n_std * np.sqrt(np.maximum(vals, 0))
    ax.add_patch(Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs))

Forget landmarks. Just track the robot's poses and the relative transforms
between them. When the robot revisits a place, add a loop closure edge. Now
optimize: find the set of poses that best satisfies all constraints
simultaneously. This is Graph SLAM, and it is how most real systems work.

This chapter implements **pose graph optimization** from scratch using
Gauss-Newton. We build the graph, add noisy odometry edges and a loop
closure edge, and watch the trajectory snap into place.

```{admonition} What you will build
:class: tip

- Implement 2D pose graph SLAM from scratch using Gauss-Newton optimization
- Build a pose graph from noisy odometry and a loop closure constraint
- Watch the trajectory snap into place when the loop closure is added and the graph is optimized
- Understand how the optimization distributes the correction across all poses

**Real world application:** Pose graph optimization is how most production SLAM systems work (Cartographer, ORB-SLAM, LIO-SAM). After this chapter, you will have built the core algorithm used in real self driving cars and mapping robots.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **Cartographer (Google)** | Production pose graph SLAM for LiDAR, used in many robots |
| **slam_toolbox (ROS 2)** | Online/offline pose graph SLAM for 2D LiDAR |
| **g2o / GTSAM** | The back end optimizers used by most pose graph SLAM systems |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## 25.1 Pose Graph: Nodes = Poses, Edges = Relative Transforms

In a **pose graph**, the state contains only robot poses:

$$\mathbf{X} = \{\mathbf{x}_0, \mathbf{x}_1, \ldots, \mathbf{x}_T\}, \qquad
\mathbf{x}_i = \begin{bmatrix} x_i \\ y_i \\ \theta_i \end{bmatrix}$$

Each edge $e_{ij}$ stores a **relative measurement** $\mathbf{z}_{ij}$:
the transform from pose $i$ to pose $j$ as measured by odometry or
loop closure.

In [ ]:
def pose2_compose(a, b):
    """Compose two 2D poses: a (+) b."""
    c = np.cos(a[2])
    s = np.sin(a[2])
    return np.array([
        a[0] + c * b[0] - s * b[1],
        a[1] + s * b[0] + c * b[1],
        a[2] + b[2]
    ])

def pose2_inverse(a):
    """Inverse of a 2D pose."""
    c = np.cos(a[2])
    s = np.sin(a[2])
    return np.array([
        -c * a[0] - s * a[1],
         s * a[0] - c * a[1],
        -a[2]
    ])

def pose2_between(a, b):
    """Relative transform from pose a to pose b: a^{-1} (+) b."""
    return pose2_compose(pose2_inverse(a), b)

def wrap_angle(a):
    """Wrap angle to [-pi, pi]."""
    return (a + np.pi) % (2 * np.pi) - np.pi

print('Pose composition utilities defined.')
print('Test: compose (1,0,0) and (1,0,pi/2):')
result = pose2_compose([1, 0, 0], [1, 0, np.pi/2])
print(f'  Result: ({result[0]:.2f}, {result[1]:.2f}, {np.degrees(result[2]):.1f} deg)')

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
# Ground truth: robot drives a rectangle
side_length = 4.0
steps_per_side = 4
sigma_odom_xy = 0.1   # odometry noise in x, y
sigma_odom_th = 0.05  # odometry noise in theta
# ──────────────────────────────────────────────────────────────────────────────

# Generate ground truth poses for a rectangle
true_poses = [np.array([0.0, 0.0, 0.0])]
step = side_length / steps_per_side

# Right side
for i in range(steps_per_side):
    p = true_poses[-1].copy()
    p[0] += step; true_poses.append(p)
# Turn left 90
p = true_poses[-1].copy(); p[2] += np.pi/2; true_poses.append(p)
# Top side
for i in range(steps_per_side):
    p = true_poses[-1].copy()
    p[0] += step * np.cos(p[2]); p[1] += step * np.sin(p[2])
    true_poses.append(p)
# Turn left 90
p = true_poses[-1].copy(); p[2] += np.pi/2; true_poses.append(p)
# Left side (going back)
for i in range(steps_per_side):
    p = true_poses[-1].copy()
    p[0] += step * np.cos(p[2]); p[1] += step * np.sin(p[2])
    true_poses.append(p)
# Turn left 90
p = true_poses[-1].copy(); p[2] += np.pi/2; true_poses.append(p)
# Bottom side (going back to start)
for i in range(steps_per_side):
    p = true_poses[-1].copy()
    p[0] += step * np.cos(p[2]); p[1] += step * np.sin(p[2])
    true_poses.append(p)

true_poses = [p.copy() for p in true_poses]
n_poses = len(true_poses)

# Generate odometry edges with noise
edges = []  # (i, j, z_ij, Omega_ij)
odom_poses = [true_poses[0].copy()]  # dead-reckoned trajectory

for i in range(n_poses - 1):
    z_true = pose2_between(true_poses[i], true_poses[i+1])
    noise = np.array([np.random.randn() * sigma_odom_xy,
                       np.random.randn() * sigma_odom_xy,
                       np.random.randn() * sigma_odom_th])
    z_noisy = z_true + noise
    z_noisy[2] = wrap_angle(z_noisy[2])
    
    Omega = np.diag([1.0/sigma_odom_xy**2, 1.0/sigma_odom_xy**2, 1.0/sigma_odom_th**2])
    edges.append((i, i+1, z_noisy, Omega))
    
    # Dead-reckoning
    odom_poses.append(pose2_compose(odom_poses[-1], z_noisy))

odom_poses = np.array(odom_poses)
true_arr = np.array(true_poses)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(true_arr[:, 0], true_arr[:, 1], 'forestgreen', lw=2, marker='o', ms=5,
        label='Ground truth', zorder=3)
ax.plot(odom_poses[:, 0], odom_poses[:, 1], 'tomato', lw=2, marker='s', ms=4,
        label='Odometry (drifted)', zorder=4)
ax.set_aspect('equal'); ax.legend(fontsize=11)
ax.set_title(f'Pose graph: {n_poses} poses, odometry drift visible', fontsize=13)
plt.tight_layout()
plt.show()

drift = np.linalg.norm(odom_poses[-1, :2] - true_arr[-1, :2])
print(f'Final pose drift (odometry only): {drift:.3f} m')
print(f'The last pose should be near the first, but has drifted away.')

**Observation:** Pure odometry accumulates error at every step. After driving
a complete rectangle, the robot should return to its starting position, but
the dead-reckoned trajectory does not close. This is the drift that loop
closure will fix.

## 25.2 Constraints: Odometry Edges and Loop Closure Edges

Each edge defines a residual: the difference between the measured relative
transform and the one implied by the current pose estimates.

For edge $(i, j)$ with measurement $\mathbf{z}_{ij}$:

$$\mathbf{e}_{ij}(\mathbf{X}) = \mathbf{z}_{ij} \ominus (\mathbf{x}_i^{-1} \oplus \mathbf{x}_j)$$

The total cost is:

$$F(\mathbf{X}) = \sum_{(i,j)} \mathbf{e}_{ij}^T \, \Omega_{ij} \, \mathbf{e}_{ij}$$

**Odometry edges** connect consecutive poses. A **loop closure edge** connects
non-consecutive poses that correspond to the same physical location.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
sigma_loop_xy = 0.05    # loop closure is more precise than odometry
sigma_loop_th = 0.02
add_loop_closure = True  # toggle to see effect
# ──────────────────────────────────────────────────────────────────────────────

# Add loop closure: last pose to first pose
if add_loop_closure:
    z_loop_true = pose2_between(true_poses[-1], true_poses[0])
    noise_lc = np.array([np.random.randn() * sigma_loop_xy,
                          np.random.randn() * sigma_loop_xy,
                          np.random.randn() * sigma_loop_th])
    z_loop = z_loop_true + noise_lc
    z_loop[2] = wrap_angle(z_loop[2])
    Omega_loop = np.diag([1.0/sigma_loop_xy**2, 1.0/sigma_loop_xy**2, 1.0/sigma_loop_th**2])
    edges.append((n_poses - 1, 0, z_loop, Omega_loop))
    print(f'Loop closure edge added: pose {n_poses-1} -> pose 0')
    print(f'  Measured relative transform: dx={z_loop[0]:.3f}, dy={z_loop[1]:.3f}, '
          f'dtheta={np.degrees(z_loop[2]):.1f} deg')

print(f'\nTotal edges: {len(edges)}')
print(f'  Odometry: {n_poses - 1}')
print(f'  Loop closure: {1 if add_loop_closure else 0}')

## 25.3 Optimization: Gauss-Newton on the Pose Graph

To optimize, we linearize each residual and solve the normal equations.
For each edge $(i, j)$:

$$\mathbf{e}_{ij}(\mathbf{X} + \Delta\mathbf{X}) \approx \mathbf{e}_{ij} + A_{ij} \Delta\mathbf{x}_i + B_{ij} \Delta\mathbf{x}_j$$

where $A_{ij} = \frac{\partial \mathbf{e}_{ij}}{\partial \mathbf{x}_i}$ and
$B_{ij} = \frac{\partial \mathbf{e}_{ij}}{\partial \mathbf{x}_j}$.

The system becomes:

$$H \, \Delta\mathbf{x} = -\mathbf{b}$$

where $H$ and $\mathbf{b}$ accumulate contributions from all edges. We fix
the first pose (gauge freedom) by not updating it.

In [ ]:
def compute_edge_residual(xi, xj, z_ij):
    """Compute residual for edge (i,j)."""
    # Predicted relative transform
    z_pred = pose2_between(xi, xj)
    e = z_ij - z_pred
    e[2] = wrap_angle(e[2])
    return e

def compute_edge_jacobians(xi, xj):
    """Compute Jacobians A (w.r.t. xi) and B (w.r.t. xj) numerically."""
    eps = 1e-6
    z_pred_0 = pose2_between(xi, xj)
    
    A = np.zeros((3, 3))
    B = np.zeros((3, 3))
    
    for k in range(3):
        xi_plus = xi.copy(); xi_plus[k] += eps
        z_plus = pose2_between(xi_plus, xj)
        diff = z_plus - z_pred_0
        diff[2] = wrap_angle(diff[2])
        A[:, k] = -diff / eps  # negative because residual is z - z_pred
        
        xj_plus = xj.copy(); xj_plus[k] += eps
        z_plus = pose2_between(xi, xj_plus)
        diff = z_plus - z_pred_0
        diff[2] = wrap_angle(diff[2])
        B[:, k] = -diff / eps
    
    return A, B

def pose_graph_optimize(poses_init, edges, n_iterations=5, fix_first=True):
    """Gauss-Newton pose graph optimization."""
    n = len(poses_init)
    poses = [p.copy() for p in poses_init]
    cost_history = []
    
    for iteration in range(n_iterations):
        dim = 3 * n
        H = np.zeros((dim, dim))
        b = np.zeros(dim)
        total_cost = 0.0
        
        for (i, j, z_ij, Omega) in edges:
            e = compute_edge_residual(poses[i], poses[j], z_ij)
            A, B = compute_edge_jacobians(poses[i], poses[j])
            
            total_cost += e @ Omega @ e
            
            # Accumulate into H and b
            ri, rj = 3*i, 3*j
            H[ri:ri+3, ri:ri+3] += A.T @ Omega @ A
            H[ri:ri+3, rj:rj+3] += A.T @ Omega @ B
            H[rj:rj+3, ri:ri+3] += B.T @ Omega @ A
            H[rj:rj+3, rj:rj+3] += B.T @ Omega @ B
            b[ri:ri+3] += A.T @ Omega @ e
            b[rj:rj+3] += B.T @ Omega @ e
        
        cost_history.append(total_cost)
        
        # Fix first pose (gauge freedom)
        if fix_first:
            H[:3, :] = 0; H[:, :3] = 0
            H[:3, :3] = np.eye(3) * 1e6
            b[:3] = 0
        
        # Solve
        dx = np.linalg.solve(H, -b)
        
        # Update poses
        for k in range(n):
            poses[k] = poses[k] + dx[3*k:3*k+3]
            poses[k][2] = wrap_angle(poses[k][2])
    
    # Final cost
    total_cost = 0
    for (i, j, z_ij, Omega) in edges:
        e = compute_edge_residual(poses[i], poses[j], z_ij)
        total_cost += e @ Omega @ e
    cost_history.append(total_cost)
    
    return poses, cost_history

print('Pose graph optimizer (Gauss-Newton) defined.')

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_gn_iterations = 5
# ──────────────────────────────────────────────────────────────────────────────

# Initialize with odometry (dead-reckoned) poses
init_poses = [odom_poses[k].copy() for k in range(n_poses)]

opt_poses, costs = pose_graph_optimize(init_poses, edges, n_gn_iterations)
opt_arr = np.array(opt_poses)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

ax = axes[0]
ax.plot(true_arr[:, 0], true_arr[:, 1], 'forestgreen', lw=2, marker='o', ms=5,
        label='Ground truth', zorder=3)
ax.plot(odom_poses[:, 0], odom_poses[:, 1], 'tomato', lw=1.5, marker='s', ms=3,
        label='Odometry (before)', alpha=0.5, zorder=2)
ax.plot(opt_arr[:, 0], opt_arr[:, 1], 'steelblue', lw=2.5, marker='D', ms=4,
        label='Optimized (after)', zorder=4)

if add_loop_closure:
    ax.annotate('loop closure', xy=(opt_arr[-1, 0], opt_arr[-1, 1]),
                xytext=(opt_arr[-1, 0]+1, opt_arr[-1, 1]+1),
                arrowprops=dict(arrowstyle='->', color='orange', lw=2),
                fontsize=12, color='orange', fontweight='bold')

ax.set_aspect('equal'); ax.legend(fontsize=10)
ax.set_title(f'Pose graph optimization ({n_gn_iterations} GN iterations)', fontsize=13)

ax = axes[1]
ax.semilogy(costs, 'steelblue', lw=2, marker='o', ms=8)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Total cost', fontsize=12)
ax.set_title('Cost convergence', fontsize=13)

plt.tight_layout()
plt.show()

# Per-pose error
errors_before = [np.linalg.norm(odom_poses[k, :2] - true_arr[k, :2]) for k in range(n_poses)]
errors_after = [np.linalg.norm(opt_arr[k, :2] - true_arr[k, :2]) for k in range(n_poses)]
print(f'Mean position error BEFORE optimization: {np.mean(errors_before):.4f} m')
print(f'Mean position error AFTER optimization:  {np.mean(errors_after):.4f} m')
print(f'Improvement: {np.mean(errors_before)/max(np.mean(errors_after),1e-10):.1f}x')

**Observation:** The loop closure edge pulls the end of the trajectory back
to the start. The optimization distributes this correction across all
intermediate poses, producing a globally consistent trajectory. Just 3 to 5
Gauss-Newton iterations are enough for convergence.

In [ ]:
# Per-pose error comparison
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(errors_before, 'tomato', lw=2, marker='s', ms=5, label='Before optimization')
ax.plot(errors_after, 'steelblue', lw=2, marker='o', ms=5, label='After optimization')
ax.set_xlabel('Pose index', fontsize=12)
ax.set_ylabel('Position error (m)', fontsize=12)
ax.set_title('Per-pose error: before vs after optimization', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

**Key insight:** Without loop closure, errors grow monotonically with
distance from the start. With loop closure and optimization, the error
is distributed evenly across all poses. The poses near the middle of
the trajectory see the largest correction because they are farthest
from both the anchor (pose 0) and the loop closure constraint.

---

## Capstone: 2D Pose Graph SLAM from Scratch

We build a complete pose graph SLAM system. The robot drives a rectangle
with noisy odometry, receives a loop closure detection, and we optimize
with Gauss-Newton. We show the trajectory before and after optimization,
along with the information matrix structure.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(123)
n_rect_steps = 8        # steps per side
rect_side = 6.0
sigma_odom = 0.15       # odometry noise
sigma_odom_th = 0.08
sigma_lc = 0.03         # loop closure noise (much tighter)
sigma_lc_th = 0.01
n_opt_iters = 3
# ──────────────────────────────────────────────────────────────────────────────

# Generate ground truth rectangle
step_cap = rect_side / n_rect_steps
gt_poses_cap = [np.array([0.0, 0.0, 0.0])]
headings = [0, np.pi/2, np.pi, 3*np.pi/2]

for side in range(4):
    heading = headings[side]
    # Add turn if not first side
    if side > 0:
        p = gt_poses_cap[-1].copy()
        p[2] = heading
        gt_poses_cap.append(p)
    # Add steps along this side
    for s in range(n_rect_steps):
        p = gt_poses_cap[-1].copy()
        p[0] += step_cap * np.cos(heading)
        p[1] += step_cap * np.sin(heading)
        gt_poses_cap.append(p)

n_cap = len(gt_poses_cap)
gt_arr_cap = np.array(gt_poses_cap)

# Build edges with noise
edges_cap = []
odom_cap = [gt_poses_cap[0].copy()]

for i in range(n_cap - 1):
    z_true = pose2_between(gt_poses_cap[i], gt_poses_cap[i+1])
    noise = np.array([np.random.randn()*sigma_odom, np.random.randn()*sigma_odom,
                       np.random.randn()*sigma_odom_th])
    z_noisy = z_true + noise
    z_noisy[2] = wrap_angle(z_noisy[2])
    Omega = np.diag([1/sigma_odom**2, 1/sigma_odom**2, 1/sigma_odom_th**2])
    edges_cap.append((i, i+1, z_noisy, Omega))
    odom_cap.append(pose2_compose(odom_cap[-1], z_noisy))

odom_cap = np.array(odom_cap)

# Loop closure
z_lc_true = pose2_between(gt_poses_cap[-1], gt_poses_cap[0])
z_lc = z_lc_true + np.array([np.random.randn()*sigma_lc, np.random.randn()*sigma_lc,
                               np.random.randn()*sigma_lc_th])
z_lc[2] = wrap_angle(z_lc[2])
Omega_lc = np.diag([1/sigma_lc**2, 1/sigma_lc**2, 1/sigma_lc_th**2])
edges_cap.append((n_cap - 1, 0, z_lc, Omega_lc))

# Optimize
init_cap = [odom_cap[k].copy() for k in range(n_cap)]
opt_cap, costs_cap = pose_graph_optimize(init_cap, edges_cap, n_opt_iters)
opt_cap_arr = np.array(opt_cap)

print(f'Pose graph: {n_cap} poses, {len(edges_cap)} edges')
print(f'Optimized in {n_opt_iters} Gauss-Newton iterations')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Before optimization
ax = axes[0]
ax.plot(gt_arr_cap[:, 0], gt_arr_cap[:, 1], 'forestgreen', lw=1.5, ls='--', alpha=0.5, label='Ground truth')
ax.plot(odom_cap[:, 0], odom_cap[:, 1], 'tomato', lw=2, marker='o', ms=3, label='Odometry')
ax.set_aspect('equal'); ax.legend(fontsize=9)
ax.set_title('Before: odometry only (drifted)', fontsize=13)

# After optimization
ax = axes[1]
ax.plot(gt_arr_cap[:, 0], gt_arr_cap[:, 1], 'forestgreen', lw=1.5, ls='--', alpha=0.5, label='Ground truth')
ax.plot(opt_cap_arr[:, 0], opt_cap_arr[:, 1], 'steelblue', lw=2, marker='D', ms=3, label='Optimized')
ax.set_aspect('equal'); ax.legend(fontsize=9)
ax.set_title(f'After: {n_opt_iters} GN iterations (corrected)', fontsize=13)

# Information matrix
ax = axes[2]
dim_cap = 3 * n_cap
H_viz = np.zeros((dim_cap, dim_cap))
for (i, j, z, Om) in edges_cap:
    ri, rj = 3*i, 3*j
    H_viz[ri:ri+3, ri:ri+3] += 1
    H_viz[ri:ri+3, rj:rj+3] += 1
    H_viz[rj:rj+3, ri:ri+3] += 1
    H_viz[rj:rj+3, rj:rj+3] += 1
ax.spy(H_viz, aspect='auto', markersize=2, color='steelblue')
ax.set_title(f'Information matrix ({dim_cap}x{dim_cap})', fontsize=13)
# Highlight loop closure
ax.plot(0, dim_cap-2, 'o', color='orange', ms=8, zorder=10)
ax.annotate('LC', xy=(0, dim_cap-2), xytext=(10, dim_cap-10),
            arrowprops=dict(arrowstyle='->', color='orange'), fontsize=10, color='orange')

plt.suptitle('Pose Graph SLAM: the trajectory snaps into place', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed error analysis
err_before = [np.linalg.norm(odom_cap[k, :2] - gt_arr_cap[k, :2]) for k in range(n_cap)]
err_after = [np.linalg.norm(opt_cap_arr[k, :2] - gt_arr_cap[k, :2]) for k in range(n_cap)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(err_before, 'tomato', lw=2, marker='s', ms=4, label='Before (odometry)')
ax.plot(err_after, 'steelblue', lw=2, marker='o', ms=4, label='After (optimized)')
ax.set_xlabel('Pose index', fontsize=12)
ax.set_ylabel('Position error (m)', fontsize=12)
ax.set_title('Per-pose error', fontsize=13)
ax.legend()

ax = axes[1]
ax.semilogy(costs_cap, 'steelblue', lw=2, marker='o', ms=8)
ax.set_xlabel('GN iteration', fontsize=12)
ax.set_ylabel('Total cost', fontsize=12)
ax.set_title('Cost convergence', fontsize=13)

plt.tight_layout()
plt.show()

print(f'Mean error before: {np.mean(err_before):.4f} m')
print(f'Mean error after:  {np.mean(err_after):.4f} m')
print(f'Max error before:  {np.max(err_before):.4f} m')
print(f'Max error after:   {np.max(err_after):.4f} m')

**Capstone observations:**
- The odometry trajectory drifts progressively, failing to close the loop.
- Adding a single loop closure edge and running Gauss-Newton produces a globally consistent trajectory.
- The optimization distributes the correction evenly: poses near the middle of the loop receive the largest adjustment.
- The information matrix is sparse and banded, with the loop closure creating a single off-diagonal block.
- Just 3 iterations of Gauss-Newton are enough to converge. This is the power of exploiting the sparse structure.

---

## Exercises

### Exercise 25.1: Remove the loop closure

Re-run the capstone with `add_loop_closure = False`. How does the optimized
trajectory compare to the odometry trajectory? Why is optimization still
slightly helpful even without loop closure?

In [ ]:
# Your code here

### Exercise 25.2: Multiple loop closures

Add loop closure edges at the corners of the rectangle (pose at the end
of each side to the pose at the start of the same side). How does adding
more constraints affect the optimized trajectory accuracy?

In [ ]:
# Your code here

### Exercise 25.3: Noise sensitivity (challenge)

Sweep the odometry noise from sigma=0.05 to sigma=0.5 in 10 steps.
For each noise level, run the full pipeline (generate trajectory, add
loop closure, optimize). Plot mean pose error vs noise level. At what
noise level does the optimizer start to fail?

In [ ]:
# Your code here